In [ ]:
from pathlib import Path

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "backend/src/plexa/mobility/modeling").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from within the Plexa clone.")
DATA_ROOT = PROJECT_ROOT / "data" / "mobility"


In [1]:
import numpy as np, pandas as pd

RAW = str(DATA_ROOT / "Automated_Traffic_Volume_Counts_20260928.csv")
r = pd.read_csv(RAW, usecols=["RequestID", "SegmentID", "Direction", "Yr", "M", "D", "HH", "MM",
                              "Vol", "street", "fromSt", "toSt"])
r["Vol"] = pd.to_numeric(r["Vol"].astype(str).str.replace(",", "", regex=False).str.strip(), errors="coerce")
r["hour"] = pd.to_datetime(dict(year=r["Yr"], month=r["M"], day=r["D"], hour=r["HH"], minute=0))

# Hourly volume per study, segment and direction: one value (hourly study) or all four 15-min intervals
g = (r.groupby(["RequestID", "SegmentID", "Direction", "hour"])
       .agg(v=("Vol", "sum"), n=("Vol", "size")).reset_index())
g = g[g["n"].isin([1, 4])]
names = r[["RequestID", "SegmentID", "Direction", "street", "fromSt", "toSt"]].drop_duplicates(
    ["RequestID", "SegmentID", "Direction"])
g = g.merge(names, on=["RequestID", "SegmentID", "Direction"], how="left")

def corridor(s):
    s = s.astype(str).str.upper().str.replace(r"[^A-Z0-9 ]", " ", regex=True)
    s = s.str.replace(r"^\s*(NB|SB|EB|WB|N|S|E|W)\s+", "", regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()

g["ckey"] = corridor(g["street"]) + " | " + corridor(g["fromSt"]) + " | " + corridor(g["toSt"])

# Median across studies for each corridor, hour and direction
cd = g.groupby(["ckey", "hour", "Direction"])["v"].median().unstack("Direction")
print("Corridor-hours by direction label:", cd.notna().sum().to_dict())

for tw, (p1, p2) in [("NS", ("NB", "SB")), ("EW", ("EB", "WB"))]:
    if tw not in cd.columns:
        print(f"\n{tw}: not present"); continue
    t = cd[cd[tw].notna()]
    both = t[p1].notna() & t[p2].notna()
    print(f"\n{tw}: corridor-hours={len(t)} | with {p1} and {p2}={int(both.sum())} "
          f"| {p1} only={int((t[p1].notna() & t[p2].isna()).sum())} "
          f"| {p2} only={int((t[p1].isna() & t[p2].notna()).sum())}")
    if both.sum():
        s = t[both]
        ratio = s[tw] / (s[p1] + s[p2])
        print(ratio.describe(percentiles=[.1, .25, .5, .75, .9]).round(3).to_string())
        print(f"Share within 10% of 1: {((ratio - 1).abs() <= 0.10).mean():.3f}")
        print(f"Median {tw}/{p1} = {(s[tw] / s[p1]).median():.3f} | "
              f"Median {tw}/{p2} = {(s[tw] / s[p2]).median():.3f}")
        print("Corridors:", s.reset_index()["ckey"].nunique())

Corridor-hours by direction label: {'EB': 116318, 'EW': 891, 'NB': 119873, 'NS': 741, 'SB': 118398, 'WB': 114716}

NS: corridor-hours=741 | with NB and SB=0 | NB only=102 | SB only=0

EW: corridor-hours=891 | with EB and WB=0 | EB only=0 | WB only=0


In [2]:
HIST_PATH = str(DATA_ROOT / "Traffic_Volume_Counts_(Historical)_20260926.csv")      # the 31-column file
OUT = str(DATA_ROOT / "clean")

raw_ids = set(pd.read_csv(HIST_PATH, usecols=["SegmentID"])["SegmentID"])
kept_ids = set(pd.read_parquet(f"{OUT}/hist_hourly_v2.parquet", columns=["segment_id"])["segment_id"])
dropped = sorted(raw_ids - kept_ids)
print("Raw SegmentIDs:", len(raw_ids), "| in cleaned table:", len(kept_ids), "| dropped:", dropped)

q = pd.read_csv(f"{OUT}/hist_quarantine.csv.gz", usecols=["SegmentID", "q_reason"])
print(q[q["SegmentID"].isin(dropped)].groupby(["SegmentID", "q_reason"]).size().to_string())

Raw SegmentIDs: 1956 | in cleaned table: 1954 | dropped: [46664, 9001153]
SegmentID  q_reason     
46664      duplicate_key     8
9001153    duplicate_key    36


In [6]:
AUTO_RAW = str(DATA_ROOT / "Automated_Traffic_Volume_Counts_20260928.csv")
HIST_RAW = str(DATA_ROOT / "Traffic_Volume_Counts_(Historical)_20260926.csv")

a = pd.read_csv(AUTO_RAW, usecols=["SegmentID", "street", "fromSt", "toSt", "Yr"])
a = a[a["SegmentID"].isin([82920, 95270])]
print("AUTOMATED names:")
print(a.groupby(["SegmentID", "street", "fromSt", "toSt"])["Yr"].agg(["min", "max", "size"]).to_string())

h = pd.read_csv(HIST_RAW, usecols=["SegmentID", "Roadway Name", "From", "To", "Date"])
h = h[h["SegmentID"].isin([82920, 95270])].copy()
h["year"] = pd.to_datetime(h["Date"], format="%m/%d/%Y").dt.year
print("\nHISTORICAL names:")
print(h.groupby(["SegmentID", "Roadway Name", "From", "To"])["year"].agg(["min", "max", "size"]).to_string())

AUTOMATED names:
                                                                                                         min   max  size
SegmentID street                         fromSt                         toSt                                            
82920     VAN WYCK EXPRESSWAY EXIT 11 SB Van Wyck Ep Sb En Meadow Lk Rd Van Wyck Ep Sb En Meadow Lk Rd  2014  2014   672
95270     HUTCHINSON RIVER PARKWAY       BARTOW AVENUE                  PALMER AVENUE                   2013  2013   181

HISTORICAL names:
                                                                                min   max  size
SegmentID Roadway Name                   From                    To                            
82920     Van Wyck Expressway Exit 11 SB East Meadow Lake Bridge Jewel Avenue  2014  2014     9
95270     LEXINGTON AVENUE               121ST ST                120TH ST      2012  2013     9


In [5]:
C = pd.read_parquet(f"{OUT}/combined_hourly_v1.parquet",
                    columns=["hour_start", "source", "dst_issue"])
d = C[C["dst_issue"]].copy()
d["date"] = d["hour_start"].dt.date
d["hr"] = d["hour_start"].dt.hour
print("Flagged hours:", len(d), "| dates:", d["date"].nunique())
print(d.groupby(["source", "hr"]).size().to_string())
print("\nDays with a flagged hour, by source:")
print(d.groupby("source")["date"].nunique().to_string())

Flagged hours: 658 | dates: 33
source      hr
automated   1      66
            2      39
historical  1     531
            2      22

Days with a flagged hour, by source:
source
automated     32
historical    10


In [7]:
import pandas as pd
OUT = str(DATA_ROOT / "clean")

C = pd.read_parquet(f"{OUT}/combined_hourly_v1.parquet")
m = (C["source"] == "historical") & (C["segment_id"] == 82920) & C["identity_conflict"]
print("82920 historical conflict rows in combined:", int(m.sum()))
C.loc[m, "identity_conflict"] = False

# Optional: exclude DST-ambiguous hours from baselines (rows stay in the table)
DST_EXCLUDE = True
C["baseline_ok"] = C["road_type"].eq("unspecified") & ~C["identity_conflict"]
if DST_EXCLUDE:
    C["baseline_ok"] = C["baseline_ok"] & ~C["dst_issue"]

print("identity_conflict rows:", int(C["identity_conflict"].sum()))
print("baseline_ok rows:", int(C["baseline_ok"].sum()))
C.to_parquet(f"{OUT}/combined_hourly_v2.parquet", index=False)

seg = pd.read_parquet(f"{OUT}/segment_table_v2.parquet")
s = seg["segment_id"] == 82920
seg.loc[s, "conflict_rows"] = 0
seg.loc[s, "conflict_share"] = 0.0
seg.loc[s, "identity_conflict_all"] = False
print("82920 segment row:", seg.loc[s, ["lion_key_used", "conflict_rows", "hist_rows", "identity_conflict_all"]].to_string())
seg.to_parquet(f"{OUT}/segment_table_v3.parquet", index=False)

82920 historical conflict rows in combined: 48
identity_conflict rows: 12100
baseline_ok rows: 1411799
82920 segment row:      lion_key_used  conflict_rows  hist_rows  identity_conflict_all
2198     SegmentID            0.0      216.0                  False


In [8]:
import pandas as pd
OUT = str(DATA_ROOT / "clean")

C = pd.read_parquet(f"{OUT}/combined_hourly_v2.parquet",
                    columns=["road_type", "identity_conflict", "dst_issue", "baseline_ok", "source"])
print("rows:", len(C), "(expected 1429638)")
print("identity_conflict rows:", int(C["identity_conflict"].sum()), "(expected 12100)")
print("baseline_ok rows:", int(C["baseline_ok"].sum()), "(expected 1411799)")
print("baseline_ok rows with dst_issue:", int((C["baseline_ok"] & C["dst_issue"]).sum()), "(expected 0)")
print("DST rows kept in table:", int(C["dst_issue"].sum()))
rebuilt = C["road_type"].eq("unspecified") & ~C["identity_conflict"] & ~C["dst_issue"]
print("baseline_ok matches the rule:", bool((C["baseline_ok"] == rebuilt).all()))

rows: 1429638 (expected 1429638)
identity_conflict rows: 12100 (expected 12100)
baseline_ok rows: 1411799 (expected 1411799)
baseline_ok rows with dst_issue: 0 (expected 0)
DST rows kept in table: 658
baseline_ok matches the rule: True


In [1]:
import re
import pandas as pd

OUT = str(DATA_ROOT / "clean")
HIST_RAW =str(DATA_ROOT / "Traffic_Volume_Counts_(Historical)_20260926.csv")
AUTO_RAW = str(DATA_ROOT / "Automated_Traffic_Volume_Counts_20260928.csv")

ABBR = {"AVE": "AVENUE", "AV": "AVENUE", "ST": "STREET", "STR": "STREET", "RD": "ROAD",
        "BLVD": "BOULEVARD", "BL": "BOULEVARD", "PKWY": "PARKWAY", "PKY": "PARKWAY", "PKWAY": "PARKWAY",
        "DR": "DRIVE", "EXPWY": "EXPRESSWAY", "EXPY": "EXPRESSWAY", "EXP": "EXPRESSWAY",
        "HWY": "HIGHWAY", "HWAY": "HIGHWAY", "FT": "FORT", "MT": "MOUNT", "PL": "PLACE", "SQ": "SQUARE",
        "TPKE": "TURNPIKE", "BR": "BRIDGE", "BRDG": "BRIDGE", "E": "EAST", "W": "WEST", "N": "NORTH",
        "S": "SOUTH", "EB": "", "WB": "", "NB": "", "SB": ""}

def norm(s):
    s = re.sub(r"[^A-Z0-9 ]", " ", str(s).upper())
    s = re.sub(r"(\d+)(ST|ND|RD|TH)\b", r"\1", s)
    return " ".join(t for t in (ABBR.get(w, w) for w in s.split()) if t)

def same(a, b):
    a, b = norm(a), norm(b)
    if not a or not b:
        return False
    ta, tb = set(a.split()), set(b.split())
    return a == b or ta <= tb or tb <= ta or len(ta & tb) / len(ta | tb) >= 0.6

# Conflict rows in the current table, with their calendar dates
C = pd.read_parquet(f"{OUT}/combined_hourly_v2.parquet",
                    columns=["segment_id", "source", "identity_conflict", "hour_start"])
hc = C[(C["source"] == "historical") & C["identity_conflict"]].copy()
hc["Date"] = hc["hour_start"].dt.normalize()
keys = hc[["segment_id", "Date"]].drop_duplicates()
print("Historical conflict segments:", hc["segment_id"].nunique(),
      "| conflict segment-days:", len(keys))

# Recorded names on those same days, historical and automated
hr = pd.read_csv(HIST_RAW, usecols=["SegmentID", "Roadway Name", "Date"]).rename(
    columns={"SegmentID": "segment_id"})
hr["Date"] = pd.to_datetime(hr["Date"], format="%m/%d/%Y")
hr = hr.merge(keys, on=["segment_id", "Date"])
hist_names = hr.groupby("segment_id")["Roadway Name"].agg(lambda s: sorted(set(s)))

ar = pd.read_csv(AUTO_RAW, usecols=["SegmentID", "street"]).rename(columns={"SegmentID": "segment_id"})
ar = ar[ar["segment_id"].isin(hc["segment_id"].unique())]
auto_names = ar.groupby("segment_id")["street"].agg(lambda s: sorted(set(s)))

# Summary per conflict segment
span = hc.groupby("segment_id").agg(conflict_rows=("hour_start", "size"),
                                    first=("hour_start", "min"), last=("hour_start", "max"))
rows = []
for sid, r in span.iterrows():
    h = hist_names.get(sid, [])
    a = auto_names.get(sid, [])
    if not a:
        status = "no automated record"
    elif any(same(x, y) for x in h for y in a):
        status = "name agrees"
    else:
        status = "name differs"
    rows.append({"segment_id": sid, "status": status, "conflict_rows": int(r["conflict_rows"]),
                 "years": f"{r['first'].year}-{r['last'].year}",
                 "historical_names": " | ".join(h[:2]), "automated_names": " | ".join(a[:2])})
out = pd.DataFrame(rows).sort_values(["status", "conflict_rows"], ascending=[True, False])
print(out["status"].value_counts().to_string())
print("\nRows that would be reinstated if all 'name agrees' were accepted:",
      int(out.loc[out["status"] == "name agrees", "conflict_rows"].sum()))
print("\n=== name agrees ===")
print(out[out["status"] == "name agrees"].to_string(index=False))
print("\n=== name differs (first 15) ===")
print(out[out["status"] == "name differs"].head(15).to_string(index=False))
out.to_csv(f"{OUT}/conflict_review.csv", index=False)

Historical conflict segments: 47 | conflict segment-days: 395
status
no automated record    42
name differs            4
name agrees             1

Rows that would be reinstated if all 'name agrees' were accepted: 251

=== name agrees ===
 segment_id      status  conflict_rows     years historical_names automated_names
      29555 name agrees            251 2012-2012    FULTON AVENUE   FULTON AVENUE

=== name differs (first 15) ===
 segment_id       status  conflict_rows     years historical_names          automated_names
     110463 name differs            576 2012-2013        GRAND AVE                 GRAND ST
      23207 name differs            216 2016-2016   LIBERTY AVENUE           LIBERTY STREET
      95270 name differs            216 2012-2013 LEXINGTON AVENUE HUTCHINSON RIVER PARKWAY
       4340 name differs            212 2012-2012          4th Ave         ARTHUR KILL ROAD


In [2]:
import pandas as pd
OUT = str(DATA_ROOT / "clean")

C = pd.read_parquet(f"{OUT}/combined_hourly_v2.parquet")
m = (C["source"] == "historical") & (C["segment_id"] == 29555) & C["identity_conflict"]
print("rows to reinstate:", int(m.sum()), "(expected 251)")
C.loc[m, "identity_conflict"] = False
C["baseline_ok"] = C["road_type"].eq("unspecified") & ~C["identity_conflict"] & ~C["dst_issue"]
print("identity_conflict rows:", int(C["identity_conflict"].sum()), "(expected 11849)")
print("baseline_ok rows:", int(C["baseline_ok"].sum()))
C.to_parquet(f"{OUT}/combined_hourly_v3.parquet", index=False)

rows to reinstate: 251 (expected 251)
identity_conflict rows: 11849 (expected 11849)
baseline_ok rows: 1412050


In [3]:
import pandas as pd
OUT = str(DATA_ROOT / "clean")

dec = pd.DataFrame([
    (5946,   "Brooklyn",      "automated", "high",        "Coney Island Ave at Cortelyou Rd"),
    (17640,  "Queens",        "automated", "high",        "62nd Dr at Junction Blvd"),
    (36764,  "Manhattan",     "manual",    "high",        "Roosevelt Island Main St"),
    (36905,  "Manhattan",     "manual",    "high",        "Roosevelt Island Main St"),
    (45270,  "Brooklyn",      "automated", "high",        "Palmetto St"),
    (45393,  "Brooklyn",      "automated", "high",        "Cornelia St"),
    (45402,  "Brooklyn",      "automated", "high",        "Myrtle Ave"),
    (51124,  "Brooklyn",      "automated", "medium",      "Linden Blvd 78-79 St, near border"),
    (66420,  "Brooklyn",      "lion",      "medium",      "Grand St Bridge, border"),
    (114477, "Manhattan",     "manual",    "high",        "Roosevelt Island Main St"),
    (136594, "Queens",        "lion",      "medium-low",  "Nostrand Ave at Vernon Ave, from-street 145 Dr"),
    (141027, "Brooklyn",      "automated", "high",        "Shore Pkwy at Fountain Ave"),
    (141367, "Brooklyn",      "automated", "medium-low",  "Jackie Robinson Pkwy, Miller Ave to Vermont Pl"),
    (145550, "Brooklyn",      "automated", "medium",      "Linden Blvd service road at 78 St"),
    (155079, "Manhattan",     "lion",      "medium-high", "Queensboro Bridge upper roadway"),
    (158339, "Brooklyn",      "lion",      "medium-high", "Pulaski Bridge, Clay St and McGuinness end"),
], columns=["segment_id", "proposed", "basis", "confidence", "note"])

seg = pd.read_parquet(f"{OUT}/segment_table_v3.parquet", columns=["segment_id", "borough"])
C = pd.read_parquet(f"{OUT}/combined_hourly_v2.parquet", columns=["segment_id", "baseline_ok"])
rows = C.groupby("segment_id").size().rename("hourly_rows")
base = C[C["baseline_ok"]].groupby("segment_id").size().rename("baseline_rows")

chk = dec.merge(seg, on="segment_id", how="left").join(rows, on="segment_id").join(base, on="segment_id")
chk["changes"] = chk["proposed"] != chk["borough"]
print(chk[["segment_id", "borough", "proposed", "basis", "confidence", "changes",
           "hourly_rows", "baseline_rows"]].to_string(index=False))

changed = chk[chk["changes"]]
print(f"\nSegments changing: {len(changed)} | hourly rows affected: {int(changed['hourly_rows'].sum())} "
      f"| baseline rows affected: {int(changed['baseline_rows'].sum())}")
print("Segments not found in segment table:", int(chk["borough"].isna().sum()))

dec.to_csv(f"{OUT}/borough_decisions.csv", index=False)
print("\nSaved borough_decisions.csv (not applied anywhere).")

 segment_id   borough  proposed     basis  confidence  changes  hourly_rows  baseline_rows
       5946  Brooklyn  Brooklyn automated        high    False          266            266
      17640    Queens    Queens automated        high    False          183            183
      36764 Manhattan Manhattan    manual        high    False            1              1
      36905 Manhattan Manhattan    manual        high    False          238            238
      45270  Brooklyn  Brooklyn automated        high    False           47             47
      45393  Brooklyn  Brooklyn automated        high    False            2              2
      45402  Brooklyn  Brooklyn automated        high    False          614            614
      51124  Brooklyn  Brooklyn automated      medium    False          664            664
      66420    Queens  Brooklyn      lion      medium     True          737            737
     114477 Manhattan Manhattan    manual        high    False          237            237

In [4]:
import pandas as pd
OUT = str(DATA_ROOT / "clean")

seg = pd.read_parquet(f"{OUT}/segment_table_v3.parquet")
REVIEW = {155079: "Manhattan", 158339: "Brooklyn", 66420: "Brooklyn"}
UNRESOLVED = [136594]

m = seg["segment_id"].isin(REVIEW.keys())
seg.loc[m, "borough"] = seg.loc[m, "segment_id"].map(REVIEW)
seg.loc[m, "borough_source"] = "reviewed"
seg.loc[seg["segment_id"].isin(UNRESOLVED), "borough_source"] = "automated_unresolved"

check = seg[seg["segment_id"].isin(list(REVIEW) + UNRESOLVED)]
print(check[["segment_id", "borough", "borough_source", "boro_disagreement"]].to_string(index=False))
print("borough counts:", seg["borough"].value_counts(dropna=False).to_dict())
seg.to_parquet(f"{OUT}/segment_table_v4.parquet", index=False)

 segment_id   borough       borough_source  boro_disagreement
      66420  Brooklyn             reviewed               True
     136594  Brooklyn automated_unresolved               True
     155079 Manhattan             reviewed               True
     158339  Brooklyn             reviewed               True
borough counts: {'Brooklyn': 1305, 'Queens': 1215, 'Bronx': 797, 'Manhattan': 788, 'Staten Island': 316, None: 67}
